# Explorer les données et le modèle

Ce notebook travaille **hors ligne** sur le dernier snapshot (`data/<date>/`).
Il ne touche jamais la base. Prérequis : `export_snapshot.py` puis `backtest.py`.

Exécuter une cellule : `Maj + Entrée`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from snapshot import load_snapshot, usable_milestones
from baseline import Estimator
from metrics import add_error_columns, summarize, segment_table, format_table

snap = load_snapshot()
est = Estimator(snap)
print("snapshot :", snap.path.name, "|", len(snap.games), "jeux |", len(est.matcher.corpus), "ancres")

## 1. Les étiquettes : les milestones

Un milestone est un chiffre de ventes déclaré et daté. C'est notre **vérité terrain**
(le *label* en ML) : ce que le modèle doit retrouver.

In [ ]:
m = usable_milestones(snap)
m[["gameName", "platform", "source", "isEstimate", "units", "reportedAt"]].sample(10, random_state=1)

In [ ]:
# Distribution des chiffres : ils couvrent plusieurs ordres de grandeur,
# d'où l'intérêt de raisonner en échelle log (et en erreur ×facteur).
ax = np.log10(m.units[m.units > 0]).plot.hist(bins=40, figsize=(8, 3))
ax.set_xlabel("log10(unités)  —  5 = 100 k, 6 = 1 M, 7 = 10 M");

## 2. Disséquer une estimation

On recalcule l'estimation d'un jeu **à une date donnée**, avec ou sans *holdout*
(retrait de l'ancre du jeu lui-même dans le matcher).

In [ ]:
def explain(name: str, date: str | None = None, holdout: bool = True):
    g = snap.games[snap.games.name == name].iloc[0]
    ts = pd.Timestamp(date or snap.path.name, tz="UTC")
    as_of = int((ts - pd.Timestamp("1970-01-01", tz="UTC")) // pd.Timedelta(microseconds=1))
    e = est.estimate(g.id, as_of, holdout_game_id=g.id if holdout else None)
    p = e.profile
    print(f"{name} au {ts.date()} — {p.neighbours_used} voisins, cold start={p.cold_start}")
    print(f"  bande PC Boxleiter : {p.pc_boxleiter_low and round(p.pc_boxleiter_low, 1)}–{p.pc_boxleiter_high and round(p.pc_boxleiter_high, 1)} ventes/review")
    print(f"  parts PC/PS/Xbox : {p.pc_share:.0%} / {p.playstation_share:.0%} / {p.xbox_share:.0%}")
    for r in e.methods:
        print(f"  {r.platform:<12} {r.method:<42} {r.low:>12,} – {r.high:>12,}")
    for k, a in e.aggregates.items():
        print(f"  {k:<12} {'→ agrégat':<42} {a.low:>12,} – {a.high:>12,}")
    print(f"  TOTAL (estimatedToday)                                  {e.total[0]:>12,} – {e.total[1]:>12,}")
    print("  milestones de ce jeu :")
    print(m[m.gameId == g.id][["platform", "source", "units", "reportedAt"]].tail(5).to_string(index=False))

explain("Hollow Knight", "2025-12-17")

## 3. Le backtest

`reports/backtest_rows.parquet` : une ligne par milestone, avec l'estimation
recalculée à sa date et l'erreur.

In [ ]:
rows = pd.read_parquet("reports/backtest_rows.parquet")
ok = add_error_columns(rows[rows.hasEstimate].drop(columns=["estMid", "logErr", "factor", "within1_5", "within2", "covered", "below", "above"], errors="ignore"))
print(format_table(pd.DataFrame([summarize(ok)])))

In [ ]:
# Histogramme de l'erreur en log10 : 0 = parfait, à droite = surestimation.
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(np.log10(ok.estMid / ok.units).clip(-2, 2), bins=60)
for f, style in [(1.5, ":"), (2, "--")]:
    for sign in (-1, 1):
        ax.axvline(sign * np.log10(f), color="k", linestyle=style)
ax.set_xlabel("log10(estimation / réel)   (pointillés : ×1,5 et ×2)");

In [ ]:
# Réel vs estimé, échelle log-log. Sur la diagonale = parfait.
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(ok.units, ok.estMid, s=4, alpha=0.3)
lim = [1e3, 2e8]
ax.plot(lim, lim, "k-")
ax.plot(lim, [2 * x for x in lim], "k--"); ax.plot(lim, [x / 2 for x in lim], "k--")
ax.set(xscale="log", yscale="log", xlim=lim, ylim=lim, xlabel="réel (milestone)", ylabel="estimé (milieu de fourchette)");

## 4. Premier pas de ML : recalibrer sans tricher

Le backtest montre deux défauts :

- un **biais** : le modèle sous-estime en médiane ;
- des **fourchettes trop étroites** : la couverture est loin de 80 %.

Le réflexe naïf serait de corriger avec les chiffres du backtest, puis de
remesurer sur ces mêmes chiffres. C'est de la **triche** : on évaluerait sur les
données qui ont servi à régler. Le bon protocole :

1. **découper par jeu** : 70 % des jeux pour apprendre (*train*), 30 % pour juger (*test*) ;
2. apprendre deux paramètres sur *train* uniquement :
   - `c` = correction du biais ;
   - `s` = demi-largeur de fourchette qui couvre 80 % des cas.

   C'est une **prédiction conforme** (*split conformal prediction*) ;
3. mesurer sur *test*, que le réglage n'a jamais vu.

Ce modèle n'a que deux paramètres. C'est le plus petit modèle de ML
possible, et il illustre déjà tout le protocole.

In [ ]:
rng = np.random.default_rng(42)
games = ok.gameId.unique()
test_games = set(rng.choice(games, size=int(0.3 * len(games)), replace=False))
train, test = ok[~ok.gameId.isin(test_games)], ok[ok.gameId.isin(test_games)]

# 1) biais appris sur train
log_err = np.log(train.estMid / train.units)
c = np.exp(-np.median(log_err))
# 2) largeur conforme : 80e percentile de l'erreur résiduelle sur train
resid = np.abs(log_err + np.log(c))
s = np.exp(np.quantile(resid, 0.80))
print(f"appris sur train ({train.gameId.nunique()} jeux) : correction ×{c:.2f}, fourchette = milieu ÷/× {s:.2f}")

def recalibrate(df):
    # Fourchette symétrique en log : son centre est `mid` (pas (low+high)/2).
    mid = df.estMid * c
    return add_error_columns(df.assign(estPoint=mid, estLow=mid / s, estHigh=mid * s))

before = summarize(test)
after = summarize(recalibrate(test))
print(format_table(pd.DataFrame([
    {"test (jeux jamais vus)": "modèle actuel", **before},
    {"test (jeux jamais vus)": "+ recalibration", **after},
])))

**Lecture :** la recalibration corrige le biais et rend la couverture honnête (≈ 80 %).
Le prix : des fourchettes beaucoup plus larges. Elle ne rend pas le modèle plus *précis*
(le facteur médian bouge peu). Pour progresser vraiment, il faut de meilleures
**features** (âge du jeu, date de mesure des reviews, CCU, signaux console…) et un vrai
modèle appris. Pistes dans le README.

**À vous :** changer la graine (`42`), la proportion test, ou le quantile `0.80`,
et observer la stabilité des résultats.